# Football JEPA — V-JEPA 2.1 action-spotting runner
Thin Kaggle runner: the real code lives in the GitHub repo (`src/`, `scripts/`).
This notebook clones it, then runs **extract → train → spot** on V-JEPA 2.1 features.

**Before running:** GPU + Internet ON, attach dataset `daniels02/3-matches`, and attach the `SOCCERNET_PWD` secret (Add-ons → Secrets).

## 0. Clone repo + install deps

In [ ]:
import os
REPO = "https://github.com/DanielSch02/Football_JEPA.git"
# Always start from a clean checkout so a stale clone can't run old code.
%cd /kaggle/working
!rm -rf Football_JEPA
!git clone {REPO}
%cd Football_JEPA
!git log --oneline -1   # confirm you're on the latest commit
!pip install -q SoccerNet scipy scikit-learn matplotlib torchcodec decord


## 1. Confirm GPU + secret

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from kaggle_secrets import UserSecretsClient
print("SOCCERNET_PWD secret present:", bool(UserSecretsClient().get_secret("SOCCERNET_PWD")))


## 2. Data layout
The `3-matches` dataset is **read-only** at `/kaggle/input/3-matches` (videos + labels).
We extract V-JEPA features into the writable `/kaggle/working/soccernet`, copying the
`Labels-v2.json` alongside so train/spot find labels + features in one place.

In [ ]:
import shutil
from pathlib import Path
from footy.config import VJEPA_GAME_PATHS, default_data_dir

INPUT_DIR = default_data_dir()               # auto-detected read-only input (videos + labels)
WORK_DIR  = "/kaggle/working/soccernet"      # writable: features + labels
print("INPUT_DIR:", INPUT_DIR)

for game in VJEPA_GAME_PATHS:
    src_lab = Path(INPUT_DIR) / game / "Labels-v2.json"
    dst_lab = Path(WORK_DIR) / game / "Labels-v2.json"
    dst_lab.parent.mkdir(parents=True, exist_ok=True)
    if src_lab.exists() and not dst_lab.exists():
        shutil.copy(src_lab, dst_lab)
    # also copy the ResNet features if present, so a baseline comparison is possible
    for half in (1, 2):
        rn = Path(INPUT_DIR) / game / f"{half}_ResNET_TF2.npy"
        if rn.exists():
            d = Path(WORK_DIR) / game / rn.name
            if not d.exists():
                shutil.copy(rn, d)
print("Prepared working dir:", WORK_DIR)
!ls -R {WORK_DIR} | head -40

## 3. Extract V-JEPA 2.1 features
Reads `*_224p.mkv` from the input dataset, writes `*_VJEPA21_L.npy` to the working dir.
Resumable: halves whose `.npy` already exists are skipped. **This is the slow step.**

In [ ]:
!python -m scripts.extract_vjepa --data_dir /kaggle/input/3-matches --out_dir /kaggle/working/soccernet --batch 8


## 4. Train the attentive probe on V-JEPA features

In [ ]:
!python -m scripts.train --data_dir /kaggle/working/soccernet --feature_tag VJEPA21_L --epochs 30


## 5. Spotting + demo query (V-JEPA checkpoint)

In [ ]:
!python -m scripts.spot --data_dir /kaggle/working/soccernet --feature_tag VJEPA21_L     --game "england_epl/2014-2015/2015-04-11 - 19-30 Burnley 0 - 1 Arsenal" --half 1 --query Corner


## 6. (Optional) Persist features as a Kaggle Dataset
Save this notebook version, then create a Dataset from `/kaggle/working/soccernet`
so future sessions skip the slow extraction step.